In [14]:
import jax
import jax.numpy as jnp
import jax.random as random
from jax import lax, jit, vmap, grad, value_and_grad
from jax.experimental.pjit import pjit
from jax.experimental import mesh_utils
from jax.sharding import Mesh, PartitionSpec, PositionalSharding
from functools import partial
import time
import json
import os
from collections import defaultdict
import numpy as np
from typing import Dict, List, Tuple

# --- Configuration (Original + Extended)
MAX_RECURSION_DEPTH = 50
DIMENSIONAL_CONSTRAINT = 0.8
RECURSION_DEPTHS = list(range(10, MAX_RECURSION_DEPTH + 1, 10))  # [10, 20, 30, 40, 50]
DRIVE_WEIGHTS = (
    ("hunger", 0.7),
    ("shelter", 0.6),
    ("knowledge", 0.9),
    ("curiosity", 0.8),
    ("influence", 0.5)
)  # Tuple of (key, value) pairs for hashability
MEMORY_DIR = "agi_memory"
WORLD_STATE_FILE = "world_state.json"
ACTION_SPACE = ("move", "mine", "craft")  # Tuple for hashability
SENSORY_TYPES = ("block", "object")  # Tuple for hashability
MAX_MEMORY_SIZE = 1000  # Limit memory entries
REFLECTION_LOG_FILE = "reflection_log.json"

# Ensure memory directory exists
if not os.path.exists(MEMORY_DIR):
    os.makedirs(MEMORY_DIR)

# --- Original DPPU Functions
@jit
def dynamic_pi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return jnp.pi * jnp.log1p(depth + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

@jit
def dynamic_phi(depth, scale_factor):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)
    return (1 + jnp.sqrt(5)) / 2 * jnp.exp(-depth / (scale_factor + 1)) * DIMENSIONAL_CONSTRAINT

@partial(jit, static_argnames=["depth"])
def dppu_with_dynamic_pi_phi(x, depth=10, scale_factor=1.0):
    depth = jnp.minimum(depth, MAX_RECURSION_DEPTH)

    def body_fn(i, val):
        pi_dyn = dynamic_pi(i, scale_factor)
        phi_dyn = dynamic_phi(i, scale_factor)
        scale = jnp.log1p(i + 1) * scale_factor * DIMENSIONAL_CONSTRAINT

        new_val = jnp.sin(val * scale * pi_dyn) * jnp.exp(-val / (phi_dyn + 1))
        return new_val

    return lax.fori_loop(0, depth, body_fn, x)

# --- Sharding Setup (Fixed for TPU v5e-1)
devices = jax.devices()  # Automatically detect TPU devices
sharding = PositionalSharding(devices)  # Adjust sharding dynamically

batch_size = 50_000
data_size = 50_000
batch_input = jnp.linspace(0, 10, data_size)
batch_input = jax.device_put(batch_input, sharding)

batched_dppu_processing = pjit(
    lambda arr: vmap(
        lambda xi: dppu_with_dynamic_pi_phi(xi, depth=10, scale_factor=0.5), in_axes=0
    )(arr),
    in_shardings=(sharding,),
    out_shardings=sharding,
)

# --- Optimize TPU Workloads By Splitting Computations into Depth=10 Segments
def split_into_depth_10(x, total_depth):
    """Runs dppu_with_dynamic_pi_phi in Depth=10 chunks instead of deeper recursion."""
    iterations = total_depth // 10
    for _ in range(iterations):
        x = dppu_with_dynamic_pi_phi(x, depth=10)
    return x

# --- Goal Autonomy Module
@partial(jit, static_argnames=["drives"])
def compute_drive_scores(drives: Tuple[Tuple[str, float], ...], state: jnp.ndarray) -> jnp.ndarray:
    """Compute weighted drive scores based on current state."""
    weights = jnp.array([weight for _, weight in drives])
    # Use min(len(drives), state.shape[0]) to avoid index errors
    n_drives = len(drives)
    state_subset = state[:n_drives] if state.shape[0] >= n_drives else jnp.pad(state, (0, n_drives - state.shape[0]))
    scores = weights * state_subset
    return scores / jnp.sum(scores + 1e-8)  # Normalize, avoid division by zero

@partial(jit, static_argnames=["drives"])
def recursive_priority_selector(drives: Tuple[Tuple[str, float], ...], state: jnp.ndarray) -> Tuple[jnp.ndarray, int]:
    """Select highest-priority drive recursively."""
    scores = compute_drive_scores(drives, state)
    selected_idx = jnp.argmax(scores)
    return scores, selected_idx

# --- Persistent Memory Layer
class PersistentMemory:
    def __init__(self, memory_dir: str = MEMORY_DIR):
        self.memory_dir = memory_dir
        self.working = {}  # Short-term key-value store
        self.episodic = []  # List of events
        self.semantic = defaultdict(list)  # Facts grouped by type
        self.identity = {"narrative": "AGI v0.1"}  # Core identity
        self._load_memory()

    def _load_memory(self):
        """Load memory from JSON files if they exist."""
        for mem_type in ["episodic", "semantic", "identity"]:
            file_path = os.path.join(self.memory_dir, f"{mem_type}.json")
            if os.path.exists(file_path):
                with open(file_path, "r") as f:
                    data = json.load(f)
                    if mem_type == "episodic":
                        self.episodic = data
                    elif mem_type == "semantic":
                        self.semantic.update(data)
                    elif mem_type == "identity":
                        self.identity.update(data)

    def _save_memory(self):
        """Save memory to JSON files."""
        for mem_type, data in [
            ("episodic", self.episodic),
            ("semantic", dict(self.semantic)),
            ("identity", self.identity)
        ]:
            file_path = os.path.join(self.memory_dir, f"{mem_type}.json")
            with open(file_path, "w") as f:
                json.dump(data, f)

    def add_working(self, key: str, value: any):
        self.working[key] = value

    def add_episodic(self, event: Dict):
        self.episodic.append(event)
        if len(self.episodic) > MAX_MEMORY_SIZE:
            self.episodic.pop(0)
        self._save_memory()

    def add_semantic(self, fact_type: str, fact: any):
        self.semantic[fact_type].append(fact)
        if len(self.semantic[fact_type]) > MAX_MEMORY_SIZE:
            self.semantic[fact_type].pop(0)
        self._save_memory()

    def update_identity(self, key: str, value: any):
        self.identity[key] = value
        self._save_memory()

    def query(self, mem_type: str, key: str = None) -> any:
        if mem_type == "working":
            return self.working.get(key)
        elif mem_type == "episodic":
            return self.episodic
        elif mem_type == "semantic":
            return self.semantic.get(key, [])
        elif mem_type == "identity":
            return self.identity.get(key)

# --- Sensor ↔ Action Loop
@partial(jit, static_argnames=["sensory_types"])
def process_sensory_input(sensory_data: jnp.ndarray, sensory_types: Tuple[str, ...]) -> jnp.ndarray:
    """Process dummy sensory inputs (e.g., block/object types)."""
    # Return the sensory data as a feature vector, normalized
    return sensory_data / jnp.max(sensory_data + 1e-8)  # Normalize to avoid division by zero

@partial(jit, static_argnames=["action_space"])
def select_action(scores: jnp.ndarray, action_space: Tuple[str, ...]) -> int:
    """Select action based on drive scores."""
    # Map drive_scores to action_space size
    max_idx = jnp.argmax(scores)
    action_idx = jnp.mod(max_idx, len(action_space))
    return action_idx

# --- Embodiment Framework
class GameWorld:
    def __init__(self, world_file: str = WORLD_STATE_FILE):
        self.world_file = world_file
        self.state = self._load_world()
        self.actions_taken = 0
        self.max_actions = 1000  # Prevent infinite loops

    def _load_world(self) -> Dict:
        """Load or initialize JSON game world."""
        if os.path.exists(self.world_file):
            with open(self.world_file, "r") as f:
                return json.load(f)
        return {
            "position": [0, 0, 0],
            "blocks": {"0,0,0": "stone"},
            "objects": {},
            "time": 0
        }

    def _save_world(self):
        with open(self.world_file, "w") as f:
            json.dump(self.state, f)

    def sense(self) -> jnp.ndarray:
        """Simulate sensing nearby blocks/objects."""
        pos = self.state["position"]
        nearby = []
        for dx in [-1, 0, 1]:
            for dy in [-1, 0, 1]:
                for dz in [-1, 0, 1]:
                    key = f"{pos[0]+dx},{pos[1]+dy},{pos[2]+dz}"
                    block = self.state["blocks"].get(key, "air")
                    nearby.append(SENSORY_TYPES.index("block") if block != "air" else 0)
        return jnp.array(nearby, dtype=jnp.float32)

    def act(self, action_idx: int) -> Tuple[float, bool]:
        """Execute action and return reward, success."""
        action = ACTION_SPACE[action_idx]
        reward = 0.0
        success = False
        if action == "move":
            self.state["position"][0] += 1  # Simple move
            reward = 0.1
            success = True
        elif action == "mine":
            pos = self.state["position"]
            key = f"{pos[0]},{pos[1]},{pos[2]}"
            if self.state["blocks"].get(key):
                del self.state["blocks"][key]
                reward = 0.5
                success = True
        elif action == "craft":
            reward = 0.3
            success = True
        self.state["time"] += 1
        self.actions_taken += 1
        self._save_world()
        return reward, success

    def get_feedback(self) -> Dict:
        """Return feedback: success, pain, reward, time pressure."""
        time_pressure = self.state["time"] / 1000.0
        return {
            "success": self.actions_taken > 0,
            "pain": 0.0 if self.actions_taken < self.max_actions else 1.0,
            "reward": 0.0,  # Updated per action
            "time_pressure": time_pressure
        }

# --- Self-Reflection Layer
class ReflectionLog:
    def __init__(self, log_file: str = REFLECTION_LOG_FILE):
        self.log_file = log_file
        self.log = self._load_log()

    def _load_log(self) -> List[Dict]:
        if os.path.exists(self.log_file):
            with open(self.log_file, "r") as f:
                return json.load(f)
        return []

    def _save_log(self):
        with open(self.log_file, "w") as f:
            json.dump(self.log, f)

    def add_entry(self, thought: str, action: str, outcome: Dict):
        self.log.append({"thought": thought, "action": action, "outcome": outcome})
        if len(self.log) > MAX_MEMORY_SIZE:
            self.log.pop(0)
        self._save_log()

    @partial(jit, static_argnums=(0, 1))
    def evaluate_alignment(self, drives: Tuple[Tuple[str, float], ...], state: jnp.ndarray, outcome: Dict) -> float:
        """Evaluate if action aligned with goals."""
        scores = compute_drive_scores(drives, state)
        alignment = outcome["reward"] * jnp.sum(scores)
        return alignment

# --- Integrate into Main Loop
memory = PersistentMemory()
world = GameWorld()
reflection = ReflectionLog()

# Update state with sensory and memory data
def update_state_with_sensory_memory(batch_input: jnp.ndarray) -> jnp.ndarray:
    sensory_data = world.sense()
    sensory_features = process_sensory_input(sensory_data, SENSORY_TYPES)
    # Reshape sensory_features to be compatible with batch_input
    sensory_features = jnp.expand_dims(sensory_features, axis=0)  # Shape: (1, 27)
    # Concatenate along a new axis or adjust dimensions as needed
    state = jnp.concatenate([batch_input[:27], sensory_features[0]], axis=0)  # Truncate batch_input for simplicity
    return state

# Main AGI loop
def agi_loop(batch_input: jnp.ndarray, depth: int = 10):
    state = update_state_with_sensory_memory(batch_input)

    # Goal Autonomy
    drive_scores, selected_drive = recursive_priority_selector(DRIVE_WEIGHTS, state)
    memory.add_episodic({"drive_selected": [name for name, _ in DRIVE_WEIGHTS][selected_drive]})

    # Action Selection
    action_idx = select_action(drive_scores, ACTION_SPACE)
    reward, success = world.act(int(action_idx))  # Convert to Python int

    # Feedback and Memory Update
    feedback = world.get_feedback()
    feedback["reward"] = reward
    memory.add_semantic("action", {"action": ACTION_SPACE[int(action_idx)], "reward": reward})
    memory.update_identity("last_action", ACTION_SPACE[int(action_idx)])

    # Self-Reflection
    thought = f"Selected drive: {[name for name, _ in DRIVE_WEIGHTS][selected_drive]}"
    alignment = reflection.evaluate_alignment(DRIVE_WEIGHTS, state, feedback)
    reflection.add_entry(thought, ACTION_SPACE[int(action_idx)], {"reward": reward, "alignment": float(alignment)})

    # Process with existing DPPU
    if depth == 10:
        output_batch = batched_dppu_processing(batch_input)
    else:
        output_batch = split_into_depth_10(batch_input, depth)

    return output_batch, feedback

# --- Modified Main Execution
NUM_TRIALS = 10
INPUT_SIZE = 50_000

# Warm-up compile
_ = dppu_with_dynamic_pi_phi(jnp.ones((INPUT_SIZE,)), depth=10)

for depth in RECURSION_DEPTHS:
    output_batch, feedback = agi_loop(batch_input, depth)
    print(f"Batch Output Shape (Depth={depth}):", output_batch.shape)
    print(f"Feedback (Depth={depth}):", feedback)

# --- Benchmarking (Updated)
for depth in RECURSION_DEPTHS:
    times = []
    for _ in range(NUM_TRIALS):
        start = time.time()
        result, feedback = agi_loop(jnp.ones((INPUT_SIZE,)), depth)
        _ = jax.device_get(result)
        end = time.time()
        times.append(end - start)

    avg_time = sum(times) / len(times)
    print(f"\n🔥 TPU Benchmark (Depth={depth}, Size={INPUT_SIZE})")
    print(f"Avg: {avg_time:.6f}, Min: {min(times):.6f}, Max: {max(times):.6f}")
    print(f"Last Feedback:", feedback)

print("\n🚀 TPU Model:", jax.devices()[0].device_kind)

# --- XLA Compilation Debugging (Optional)
# Pre-compile with fixed depth to avoid tracing static arguments
compiled_fn_10 = jax.jit(lambda x: dppu_with_dynamic_pi_phi(x, depth=10)).lower(jnp.ones((50_000,)))
compiled_fn_20 = jax.jit(lambda x: dppu_with_dynamic_pi_phi(x, depth=20)).lower(jnp.ones((50_000,)))

print("\n🚀 XLA Compilation for Depth=10:")
print(compiled_fn_10.as_text())

print("\n🚀 XLA Compilation for Depth=20:")
print(compiled_fn_20.as_text())

Batch Output Shape (Depth=10): (50000,)
Feedback (Depth=10): {'success': True, 'pain': 0.0, 'reward': 0.1, 'time_pressure': 0.058}
Batch Output Shape (Depth=20): (50000,)
Feedback (Depth=20): {'success': True, 'pain': 0.0, 'reward': 0.1, 'time_pressure': 0.059}
Batch Output Shape (Depth=30): (50000,)
Feedback (Depth=30): {'success': True, 'pain': 0.0, 'reward': 0.1, 'time_pressure': 0.06}
Batch Output Shape (Depth=40): (50000,)
Feedback (Depth=40): {'success': True, 'pain': 0.0, 'reward': 0.1, 'time_pressure': 0.061}
Batch Output Shape (Depth=50): (50000,)
Feedback (Depth=50): {'success': True, 'pain': 0.0, 'reward': 0.1, 'time_pressure': 0.062}

🔥 TPU Benchmark (Depth=10, Size=50000)
Avg: 0.015836, Min: 0.004480, Max: 0.116117
Last Feedback: {'success': True, 'pain': 0.0, 'reward': 0.3, 'time_pressure': 0.072}

🔥 TPU Benchmark (Depth=20, Size=50000)
Avg: 0.004767, Min: 0.004557, Max: 0.005168
Last Feedback: {'success': True, 'pain': 0.0, 'reward': 0.3, 'time_pressure': 0.082}

🔥 TPU B